# Local Text Embeddings: Step-by-Step
Learn how to generate text embeddings locally and convert sentences into vectors.

## 1) Install dependencies
We will use sentence-transformers to run a local embedding model.

In [ ]:
%pip install sentence-transformers psycopg2-binary

## 2) Load a local embedding model
This uses a small model for fast local inference.

In [39]:
from sentence_transformers import SentenceTransformer

model_name = "all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

## 3) Convert sentences into vectors
The output is a list of numeric vectors (embeddings).

In [24]:


sentences = [
    # "Any places I can find a dog?" , #0
    # "Where can I see dogs in public places?", #0
    "What place has pizza?", #0

    "The dog is on the park" , #1
    "There are puppy on the Plaza" , #2
    "The mall has it's own pizza" , #3
    "The police was patrolling on the road" , #4
    "The water is good in the beach" , #5
    "The it's raining in the forest" , #6
    "There are dogs on the bus" , #7
]


lowerSentence = [s.lower() for s in sentences]

print(lowerSentence)

embeddings = model.encode(lowerSentence)
print(embeddings)
print(embeddings.shape)

['what place has pizza?', 'the dog is on the park', 'there are puppy on the plaza', "the mall has it's own pizza", 'the police was patrolling on the road', 'the water is good in the beach', "the it's raining in the forest", 'there are dogs on the bus']
[[-0.00345207  0.0506988  -0.03276059 ...  0.03641402  0.00505028
  -0.01285796]
 [ 0.02244985 -0.04711957  0.06384178 ...  0.07726322 -0.01975152
  -0.01114014]
 [-0.08915724 -0.02812134  0.03781177 ...  0.10789294  0.02788689
   0.05764466]
 ...
 [-0.00839002 -0.01533703  0.09181897 ... -0.00146245 -0.0009897
   0.07458625]
 [ 0.01014331  0.00412862  0.06011282 ... -0.09143323 -0.0200039
   0.01955765]
 [-0.00027769 -0.03765049  0.03488282 ...  0.15971082  0.00177139
  -0.00101693]]
(8, 384)


## 4) Compare similarity between sentences
Use cosine similarity to see which sentences are closer in meaning.

In [25]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("===========Value===========")
# print("Embeddings: ", embeddings[0])
print("===========================\n")


for index, value in enumerate(embeddings):
    if index != 7:
        result = cosine_similarity(embeddings[index], embeddings[index + 1])
        print(f"Sim 0 - Sim {index + 1} : {result:.2f}")


===========Value===========

Sim 0 - Sim 1 : 0.07
Sim 0 - Sim 2 : 0.36
Sim 0 - Sim 3 : 0.25
Sim 0 - Sim 4 : 0.02
Sim 0 - Sim 5 : 0.05
Sim 0 - Sim 6 : 0.27
Sim 0 - Sim 7 : -0.01


## 5) Embed a single sentence
This is useful for live input or small lookups.

In [26]:
text = "Local embeddings are fast and private."
vector = model.encode([text])[0]
print(vector)
print(len(vector))

[ 2.60942560e-02 -7.29857311e-02  4.50087562e-02 -3.58769787e-03
  1.14214554e-01  8.29491094e-02 -6.04627244e-02 -7.00024143e-02
  4.81808744e-02 -3.40556577e-02  1.24712750e-01  4.68053147e-02
 -1.43438121e-02  2.25963332e-02 -4.69922833e-02 -1.17452245e-03
  8.27708021e-02  3.20072211e-02 -3.39860208e-02  7.03511527e-03
 -7.56127909e-02 -4.57767844e-02  4.65470366e-02 -4.09796536e-02
  1.32990524e-01 -5.89111783e-02 -1.91951524e-02  7.23462701e-02
  1.06314257e-01 -7.02424571e-02  3.43335792e-02  2.20791660e-02
  7.69197149e-03  1.27868783e-02 -4.47853794e-03  9.37895030e-02
 -3.35538201e-02  1.86644644e-02 -1.01269670e-01  1.62323173e-02
  7.37747774e-02  2.08497178e-02 -2.56531853e-02  5.38773043e-03
 -1.63772833e-02 -2.59093679e-02  2.98934653e-02  2.98373085e-02
 -8.33859295e-02 -5.78484591e-03 -1.52942752e-02 -1.41525352e-02
 -9.35616270e-02  6.19795360e-03 -7.22789764e-03 -8.61346647e-02
 -6.86613494e-04 -5.43915406e-02  2.48614568e-02  6.99958652e-02
 -1.96495559e-02 -2.47954

## 6) Database connection
Configure your PostgreSQL connection details.

In [27]:
import os
import psycopg2

# Update these values for your local setup.
db_config = {
    "host": os.getenv("PGHOST", "localhost"),
    "port": int(os.getenv("PGPORT", "5432")),
    "dbname": os.getenv("PGDATABASE", "llm_iTour"),
    "user": os.getenv("PGUSER", "postgres"),
    "password": os.getenv("PGPASSWORD", "moymoybryan"),
}

conn = psycopg2.connect(**db_config)
conn.autocommit = True
cursor = conn.cursor()
print("Connected to PostgreSQL")

Connected to PostgreSQL


## 7) Database generation queries
Create tables, insert seed data, and enable the pgvector extension.

In [ ]:
schema_sql = """
CREATE EXTENSION IF NOT EXISTS vector;

CREATE TABLE IF NOT EXISTS tourist_area (
    area_id SERIAL PRIMARY KEY,
    name TEXT NOT NULL,
    description TEXT,
    location TEXT,
    region TEXT,
    latitude DOUBLE PRECISION,
    longitude DOUBLE PRECISION,
    category TEXT,
    created_at TIMESTAMP WITHOUT TIME ZONE DEFAULT NOW()
);

CREATE TABLE IF NOT EXISTS tourist_area_vectors (
    vector_id SERIAL PRIMARY KEY,
    area_id INTEGER REFERENCES tourist_area(area_id) ON DELETE CASCADE,
    content TEXT,
    embedding VECTOR(1536),
    created_at TIMESTAMP WITHOUT TIME ZONE DEFAULT NOW()
);

CREATE TABLE IF NOT EXISTS area_activities (
    activity_id SERIAL PRIMARY KEY,
    area_id INTEGER REFERENCES tourist_area(area_id) ON DELETE CASCADE,
    activity_name TEXT,
    description TEXT,
    difficulty_level TEXT,
    estimated_duration TEXT,
    price_range TEXT,
    created_at TIMESTAMP WITHOUT TIME ZONE DEFAULT NOW()
);
"""

cursor.execute(schema_sql)
conn.commit()

print("Schema applied")

## 7.1) User history vectors
Store user query embeddings for future retrieval and personalization.

In [33]:
user_history_sql = """
CREATE TABLE IF NOT EXISTS user_history_vector (
    history_id SERIAL PRIMARY KEY,
    user_id TEXT,
    query_text TEXT NOT NULL,
    embedding VECTOR(1536),
    created_at TIMESTAMP WITHOUT TIME ZONE DEFAULT NOW()
 );
"""

cursor.execute(user_history_sql)
conn.commit()

print("User history vector table applied")

User history vector table applied


## 8) Query the database
Run a simple query to verify the data.

In [ ]:
import numpy as np
from data.seed_data import ACTIVITIES, AREAS, VECTORS

areas = AREAS
activities = ACTIVITIES
vectors = VECTORS


def get_or_create_area(area):
    cursor.execute("SELECT area_id FROM tourist_area WHERE name = %s;", (area["name"],))
    row = cursor.fetchone()
    if row:
        return row[0]

    cursor.execute(
        """
        INSERT INTO tourist_area (
            name, description, location, region, latitude, longitude, category
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s)
        RETURNING area_id;
        """,
        (
            area["name"],
            area["description"],
            area["location"],
            area["region"],
            area["latitude"],
            area["longitude"],
            area["category"],
        ),
    )
    return cursor.fetchone()[0]


area_ids = {area["name"]: get_or_create_area(area) for area in areas}

for activity in activities:
    area_id = area_ids[activity["area"]]
    cursor.execute(
        "SELECT 1 FROM area_activities WHERE area_id = %s AND activity_name = %s;",
        (area_id, activity["activity_name"]),
    )
    if cursor.fetchone():
        continue
    cursor.execute(
        """
        INSERT INTO area_activities (
            area_id, activity_name, description, difficulty_level, estimated_duration, price_range
        )
        VALUES (%s, %s, %s, %s, %s, %s);
        """,
        (
            area_id,
            activity["activity_name"],
            activity["description"],
            activity["difficulty_level"],
            activity["estimated_duration"],
            activity["price_range"],
        ),
    )


def normalize_embedding(vec, target_dim=1536):
    if len(vec) == target_dim:
        return vec
    if len(vec) > target_dim:
        return vec[:target_dim]
    # Pad with zeros to match the vector column size.
    return np.pad(vec, (0, target_dim - len(vec)), mode="constant")


def to_vector_literal(vec):
    return "[" + ",".join(f"{x:.6f}" for x in vec) + "]"


for item in vectors:
    area_id = area_ids[item["area"]]
    cursor.execute(
        "SELECT 1 FROM tourist_area_vectors WHERE area_id = %s AND content = %s;",
        (area_id, item["content"]),
    )
    if cursor.fetchone():
        continue

    embedding = model.encode([item["content"]])[0]
    embedding = normalize_embedding(embedding, 1536)
    vector_literal = to_vector_literal(embedding)

    cursor.execute(
        """
        INSERT INTO tourist_area_vectors (area_id, content, embedding)
        VALUES (%s, %s, %s::vector);
        """,
        (area_id, item["content"], vector_literal),
    )

conn.commit()
print("Seed data applied")

## 9) Vector search helper
Define a query filter and embedding search for tourist-related requests.

In [44]:
show_match_score = True


def is_tourist_query(query: str) -> bool:
    keywords = [
        "tourist",
        "tourism",
        "travel",
        "trip",
        "vacation",
        "itinerary",
        "visit",
        "destination",
        "spot",
        "place",
        "beach",
        "mountain",
        "island",
        "city",
        "museum",
        "park",
        "activity",
        "adventure",
        "food",
        "hotel",
    ]
    query_lower = query.lower()
    return any(word in query_lower for word in keywords)


def normalize_embedding(vec, target_dim=1536):
    if len(vec) == target_dim:
        return vec
    if len(vec) > target_dim:
        return vec[:target_dim]
    return np.pad(vec, (0, target_dim - len(vec)), mode="constant")


def to_vector_literal(vec):
    return "[" + ",".join(f"{x:.6f}" for x in vec) + "]"


def vector_search_tourist(query: str, top_k: int = 5):
    if not is_tourist_query(query):
        print("Query rejected: not related to tourist spots or activities.")
        return []

    from psycopg2.extras import RealDictCursor
    import json

    query_embedding = model.encode([query])[0]
    query_embedding = normalize_embedding(query_embedding, 1536)
    vector_literal = to_vector_literal(query_embedding)

    local_cursor = conn.cursor(cursor_factory=RealDictCursor)
    try:
        local_cursor.execute(
            """
            SELECT
                ta.area_id,
                ta.name,
                ta.description,
                ta.location,
                ta.region,
                ta.latitude,
                ta.longitude,
                ta.category,
                ta.created_at,
                tav.vector_id,
                tav.area_id AS vector_area_id,
                tav.content,
                tav.embedding,
                tav.created_at AS vector_created_at,
                tav.embedding <-> %s::vector AS distance,
                COALESCE(
                    json_agg(
                        json_build_object(
                            'activity_id', aa.activity_id,
                            'area_id', aa.area_id,
                            'activity_name', aa.activity_name,
                            'description', aa.description,
                            'difficulty_level', aa.difficulty_level,
                            'estimated_duration', aa.estimated_duration,
                            'price_range', aa.price_range,
                            'created_at', aa.created_at
                        )
                        ORDER BY aa.activity_id
                    ) FILTER (WHERE aa.activity_id IS NOT NULL),
                    '[]'::json
                ) AS activities
            FROM tourist_area_vectors tav
            JOIN tourist_area ta ON ta.area_id = tav.area_id
            LEFT JOIN area_activities aa ON aa.area_id = ta.area_id
            GROUP BY
                ta.area_id,
                ta.name,
                ta.description,
                ta.location,
                ta.region,
                ta.latitude,
                ta.longitude,
                ta.category,
                ta.created_at,
                tav.vector_id,
                tav.area_id,
                tav.content,
                tav.embedding,
                tav.created_at
            ORDER BY tav.embedding <-> %s::vector
            LIMIT %s;
            """,
            (vector_literal, vector_literal, top_k),
        )
        rows = local_cursor.fetchall()
    finally:
        local_cursor.close()

    for row in rows:
        activities = row.get("activities")
        if isinstance(activities, str):
            row["activities"] = json.loads(activities)

    if show_match_score:
        print("Match scores:")
        for row in rows:
            area_id = row.get("area_id")
            name = row.get("name")
            distance = row.get("distance")
            if distance is not None:
                print(f"{area_id} | {name} | {distance:.4f}")
            else:
                print(f"{area_id} | {name} | n/a")
    return rows


user_query = "Any extreme adventures recommend for me?"
vector_search_tourist(user_query)

Match scores:
4 | Puerto Princesa Underground River | 1.1332
1 | Boracay | 1.1513
2 | Siargao | 1.1756
5 | Cebu City | 1.1858
6 | Baguio | 1.2196


[RealDictRow([('area_id', 4),
              ('name', 'Puerto Princesa Underground River'),
              ('description', 'Protected river cave system and nature park.'),
              ('location', 'Palawan'),
              ('region', 'Mimaropa'),
              ('latitude', 10.1928),
              ('longitude', 118.9263),
              ('category', 'nature'),
              ('created_at',
               datetime.datetime(2026, 5, 11, 7, 29, 46, 351487)),
              ('vector_id', 4),
              ('vector_area_id', 4),
              ('content', 'Underground river cave tour and nature park.'),
              ('embedding',
               '[0.02073,0.021063,0.044258,0.021643,-0.050311,-0.001525,-0.008056,-0.014425,-0.014538,0.040768,-0.016724,-0.106386,-0.042765,0.058994,0.040297,0.039767,0.010551,0.058827,0.103963,-0.061091,-0.015849,0.016696,0.0116,-0.001957,-0.074587,0.022661,-0.076511,0.052663,0.038706,-0.032308,-0.011657,0.071604,0.011143,-0.063986,0.043458,0.132311,-0.010953,-0.0331

## 10) Generate a tour guide response
Use the matched places to craft a concise, reasoned reply.

In [32]:
from langchain_community.chat_models import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatOllama(model="gemma:2b", temperature=0.2)

def build_reasoned_response(query: str, rows):
    if not rows:
        return (
            "Sorry, I do not have enough data to answer that yet. "
            "Try asking about beaches, mountains, or cultural spots."
        )

    context_lines = []
    for row in rows[:3]:
        name = row.get("name")
        location = row.get("location")
        category = row.get("category")
        content = row.get("content")
        activities = row.get("activities", [])
        activity_names = ", ".join(
            activity.get("activity_name", "")
            for activity in activities[:3]
            if activity.get("activity_name")
        )
        activity_hint = f" Activities: {activity_names}." if activity_names else ""
        context_lines.append(
            f"- {name} in {location}: {content} ({category}).{activity_hint}"
        )
    context_block = "\n".join(context_lines)

    messages = [
        SystemMessage(content=
            "You are a friendly tour guide. Use the matched places to answer the user's question "
            "and give a short rationale for each suggestion. Keep it concise."
        ),
        HumanMessage(content=(
            f"User question: {query}\n"
            f"Matches:\n{context_block}\n"
            "Answer with up to 3 suggestions "
            # " and a brief rationale per suggestion. "
            "Include estimated price range budgets"
        )),
    ]
    response = llm.invoke(messages)
    return response.content.strip()


user_query = "Any travel recommendation that can be covered by 15k budget?"
rows = vector_search_tourist(user_query)
print("\n" + build_reasoned_response(user_query, rows))


**Suggestion 1: Cebu City**

* **Price range:** P10,000 - P15,000 per person
* **Rationale:** Cebu City offers a diverse mix of historical attractions, delicious food, and exciting activities within a reasonable budget. The city is easily accessible by air from Manila, making it a convenient destination.

**Suggestion 2: Boracay**

* **Price range:** P12,000 - P18,000 per person
* **Rationale:** Boracay boasts stunning beaches, island hopping tours, and a vibrant nightlife scene, providing a fun and exciting experience within a budget.

**Suggestion 3: Puerto Princesa Underground River**

* **Price range:** P15,000 - P20,000 per person
* **Rationale:** The Underground River Cave Tour in Palawan is a unique and unforgettable experience that can be enjoyed within a budget. The cave tour offers a glimpse into a fascinating natural wonder while remaining accessible to budget-conscious travelers.


## 11) Chat with Gemma + save preferences
Extract simple user preferences and store them in `user_history_vector`.

In [42]:
import json


def extract_preference_text(user_text: str) -> str:
    """Return a short preference summary or an empty string if none is found."""
    messages = [
        SystemMessage(content=(
            "Extract a single short user preference from the input. "
            "Return JSON only with keys: preference. "
            "If no clear preference, set preference to an empty string."
        )),
        HumanMessage(content=f"Input: {user_text}"),
    ]
    response = llm.invoke(messages)
    content = response.content.strip()
    try:
        data = json.loads(content)
    except json.JSONDecodeError:
        return ""
    preference = data.get("preference", "")
    if not isinstance(preference, str):
        return ""
    return preference.strip()


def save_user_preference(user_text: str, user_id: str = "demo") -> str:
    preference = extract_preference_text(user_text)
    if not preference:
        return "No preference detected."

    embedding = model.encode([preference])[0]
    embedding = normalize_embedding(embedding, 1536)
    vector_literal = to_vector_literal(embedding)

    cursor.execute(
        """
        INSERT INTO user_history_vector (user_id, query_text, embedding)
        VALUES (%s, %s, %s::vector);
        """,
        (user_id, preference, vector_literal),
    )
    conn.commit()
    return f"Saved preference: {preference}"


user_input = "What is the capita of the philippines?"
print(save_user_preference(user_input))

# Optional: ask the model to respond like a tour guide
rows = vector_search_tourist(user_input)
print("\n" + build_reasoned_response(user_input, rows))

Saved preference: What is the capita of the Philippines?
Query rejected: not related to tourist spots or activities.

Sorry, I do not have enough data to answer that yet. Try asking about beaches, mountains, or cultural spots.
